Q1(0.5점)

import re

text = "강의는 2026-05-06, 숙제는 2026-05-18에 마감"

# (a) re.match: 문자열 처음부터만 매칭 시도 → 첫 글자가 한글이므로 None
print(re.match(r"\d+", text))

# (b) re.search: 문자열 전체에서 첫 번째 매칭 탐색
print(re.search(r"\d{4}-\d{2}-\d{2}", text).group())

# (c) findall, 캡처 그룹 없음 → 매칭 문자열 자체의 리스트
print(re.findall(r"\d{4}-\d{2}-\d{2}", text))

# (d) findall, 캡처 그룹 3개 → 각 매칭이 (년, 월, 일) 튜플인 리스트
print(re.findall(r"(\d{4})-(\d{2})-(\d{2})", text))

# (e) findall, 비캡처 그룹 (?:…) → (c)와 동일하게 문자열 리스트
print(re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text))

(a)None
(b)'2026-05-06'
(c)['2026-05-06', '2026-05-18']
(d)[('2026', '05', '06'), ('2026', '05', '18')]
(e)['2026-05-06', '2026-05-18']

(c), (d), (e)가 서로 다른 결과를 돌려주는 이유
re.findall은 패턴 안에 캡처 그룹 (...)이 하나라도 있으면 그룹에 잡힌 부분만 묶어 튜플로 반환한다. 
(d)는 캡처 그룹이 3개이므로 각 매칭이 (년, 월, 일) 튜플이 되는 반면, (c)와 (e)는 캡처 그룹이 없어(비캡처 그룹 (?:…)도 캡처하지 않으므로) 매칭된 전체 문자열을 그대로 반환한다.

Q2(0.5점)

import re

html = "<b>안녕</b> <i>세상</i>!"
nums = "수강생 30명, 조교 3명"

# (a) 탐욕적(greedy) .+ → 첫 '<'부터 마지막 '>'까지 한 번에 매칭
print(re.sub(r"<.+>", "[T]", html))

# (b) 게으른(lazy) .+? → '>'를 만나는 즉시 매칭 종료, 태그 개별 치환
print(re.sub(r"<.+?>", "[T]", html))

# (c) [^>]+ → '>' 이전 문자만 허용, (b)와 동일 효과
print(re.sub(r"<[^>]+>", "[T]", html))

# (d) 원시 문자열 r"<\1>" → \1이 역참조로 정상 해석
print(re.sub(r"(\d+)", r"<\1>", nums))

# (e) 일반 문자열 "<\1>" → Python이 \1을 \x01(SOH 제어문자)로 먼저 해석
print(repr(re.sub(r"(\d+)", "<\1>", nums)))

(a)'[T]!'
(b)'[T]안녕[T] [T]세상[T]!'
(c)'[T]안녕[T] [T]세상[T]!'
(d)'수강생 <30>명, 조교 <3>명'
(e)'수강생 <\x01>명, 조교 <\x01>명'

추가질문
(i) (a)의 탐욕적 수량자 .+는 가능한 한 길게 매칭하므로 첫 <부터 마지막 >까지 전체를 삼켜 [T] 하나로 치환하는 반면, (b)의 게으른 수량자 .+?는 >를 만나는 즉시 매칭을 끝내므로 각 태그가 개별적으로 치환된다.
(ii) r"<\1>"은 원시 문자열이라 \1이 정규식 역참조로 올바르게 해석되지만, "<\1>"는 일반 문자열이라 Python 인터프리터가 \1을 먼저 ASCII 제어 문자 \x01로 바꾼 뒤 re 엔진에 전달하므로 역참조로 동작하지 않는다.

Q3(1점)

In [2]:
import re
from collections import Counter

# 동일 패턴을 두 번 이상 사용하므로 미리 컴파일
RE_URL     = re.compile(r'https?://\S+')
RE_HTML    = re.compile(r'<[^>]+>')
RE_EMAIL   = re.compile(r'[\w.+-]+@[\w.-]+\.\w+')
RE_PHONE   = re.compile(r'\d{2,4}-\d{3,4}-\d{4}')
RE_MENTION = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_JAMO    = re.compile(r'[\u3131-\u3163]+')
RE_SPACES  = re.compile(r'\s+')

In [3]:
posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "  여러 공백과\n\n\n줄바꿈이 많은 텍스트  ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

In [4]:
def clean_post(post: str) -> str:
    post = RE_URL.sub(' ', post)          # 1. URL 제거
    post = RE_HTML.sub('', post)          # 2. HTML 태그 제거
    post, _ = RE_EMAIL.subn('[이메일]', post)  # 3. 이메일 마스킹
    post, _ = RE_PHONE.subn('[전화]', post)    # 3. 전화번호 마스킹
    post = RE_MENTION.sub(' ', post)      # 4. 멘션 제거
    post = RE_HASHTAG.sub(' ', post)      # 4. 해시태그 제거
    post = RE_JAMO.sub('', post)          # 5. 한글 자음/모음 제거
    post = RE_SPACES.sub(' ', post).strip()   # 6. 공백 정리
    return post

for i, p in enumerate(posts):
    print(f"[{i}] {clean_post(p)!r}")

[0] '오늘 수업 진짜 재밌었음!! 감사 자료:'
[1] '팀플 어디서 모이지 카톡'
[2] '중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!'
[3] '여러 공백과 줄바꿈이 많은 텍스트'
[4] '진짜 좋다'


In [5]:
def extract_hashtags(post: str) -> list[str]:
    return re.findall(r'#(\w+)', post)

for i, p in enumerate(posts):
    print(f"[{i}] {extract_hashtags(p)}")

[0] ['파이썬']
[1] ['DCCP2026', '팀플']
[2] []
[3] []
[4] ['파이썬', '추천']


In [6]:
def analyze_posts(posts: list[str]) -> dict:
    cleaned = [clean_post(p) for p in posts]
    avg_length_after_clean = round(sum(len(c) for c in cleaned) / len(cleaned), 2)

    all_tags: list[str] = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    hashtag_counts = dict(Counter(all_tags).most_common())

    masked_count = 0
    for p in posts:
        _, n_email = RE_EMAIL.subn('[이메일]', p)
        _, n_phone = RE_PHONE.subn('[전화]', p)
        masked_count += n_email + n_phone

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_length_after_clean,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }

print(analyze_posts(posts))

{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


6단계의 순서는 반드시 지켜야 한다. 가장 중요한 예시는 단계 3(마스킹)과 단계 4(멘션·해시태그 제거)의 순서다. 이메일 주소 mam3b@snu.ac.kr에는 @가 포함되어 있어 단계 4를 먼저 실행하면 RE_MENTION(@\w+)이 @snu를 멘션으로 오인해 이메일을 잘라낸다. 이후 단계 3에서 패턴이 맞지 않아 마스킹이 실패하고 개인정보가 그대로 노출된다. 마찬가지로 단계 1(URL 제거)을 먼저 하지 않으면 URL 안의 #이나 @가 단계 4에서 해시태그·멘션으로 잘못 처리될 수 있다.

해당 과제는 Claude.ai의 도움을 받아 작성되었습니다.
https://claude.ai/share/70c500d3-d4b7-4033-b163-a60f8687c3cf